# Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
from openai import OpenAI

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

client = OpenAI()

import warnings
warnings.filterwarnings('ignore')

In [3]:
llm_model = "gpt-3.5-turbo"

In [4]:
# pip install pandas

In [5]:
import pandas as pd
df = pd.read_csv('assets/Data.csv')

In [6]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [8]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [9]:
prompt = ChatPromptTemplate.from_template("What is the best name to describe a company that makes {product}?")

In [10]:
product = "Queen Size Sheet Set"

In [11]:
text = prompt.format(product=product)
response = llm.invoke(text)
print(response.content)

"Royal Comfort Linens"


In [12]:
# equivalently:
chain = prompt | llm
response = chain.invoke(product)
print(response.content)

"Royal Comfort Linens"


## SimpleSequentialChain

In [13]:
llm = ChatOpenAI(temperature=0.8, model=llm_model)

In [14]:
# prompt template 1
first_prompt = ChatPromptTemplate.from_template("What is the best name to describe a company that makes {product}?")

# Chain 1
chain_one = first_prompt | llm

In [15]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template("Write a 20 words description for the following company:{company_name}")

# Chain 2
chain_two = second_prompt | llm

In [16]:
from langchain_core.output_parsers import StrOutputParser

overall_simple_chain = chain_one | StrOutputParser() | chain_two

The `StrOutputParser()` is needed between the two chains because chain_one outputs an `AIMessage` object, but chain_two's prompt expects a plain `string`.
`StrOutputParser()` extracts the `.content` string from the message.

In [17]:
result = overall_simple_chain.invoke(product)
print(result.content)

Regal Linens Co. offers luxurious bedding and home textiles, combining elegant designs with exceptional quality for a truly regal experience.


## SequentialChain

In [18]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template("Translate the following review to english:\n\n{Review}")

# chain 1
chain_one = first_prompt | llm | StrOutputParser()

In [19]:
second_prompt = ChatPromptTemplate.from_template("Can you summarize the following review in 1 sentence: \n\n{English_Review}")

# chain 2
chain_two = second_prompt | llm | StrOutputParser()

In [20]:
# prompt template 3: detect language
third_prompt = ChatPromptTemplate.from_template("What language is the following review:\n\n{Review}")

# chain 3
chain_three = third_prompt | llm | StrOutputParser()

In [21]:
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)

# chain 4
chain_four = fourth_prompt | llm | StrOutputParser()

In [22]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

overall_chain = (
    RunnablePassthrough.assign(English_Review=chain_one) 
    | RunnableParallel(
        English_Review=lambda x: x["English_Review"],
        summary=lambda x: chain_two.invoke({"English_Review": x["English_Review"]}),
        language=lambda x: chain_three.invoke({"Review": x["Review"]})
    )
    | RunnablePassthrough.assign(
        followup_message=lambda x: chain_four.invoke({
            "summary": x["summary"],
            "language": x["language"]
        })
    )
)

**Step 1:** `RunnablePassthrough.assign(English_Review=chain_one)`
  
You start with:
`{"Review": "Je suis fatigué"}`

`chain_one` translates it to English. `.assign()` adds the result as a new key without removing the original:

`{"Review": "Je suis fatigué", "English_Review": "I am tired"}`

---
**Step 2:** `RunnableParallel(...)`
  
Takes the dict from step 1 and runs 3 things simultaneously:

- `English_Review=lambda x: x["English_Review"]` — just copies "I am tired" (keeps it in output)
- `summary=lambda x: chain_two.invoke(...)` — summarizes "I am tired" → "The reviewer is tired"
- `language=lambda x: chain_three.invoke(...)` — detects language of original → "French"

All three run at the same time. Result:
```json
{
    "English_Review": "I am tired",
    "summary": "The reviewer is tired",
    "language": "French"
}
```

---
**Step 3:** `RunnablePassthrough.assign(followup_message=...)`
  
Takes the dict from step 2. Runs chain_four with summary="The reviewer is tired" and language="French". Adds the result as a new key:
```json
{
    "English_Review": "I am tired",
    "summary": "The reviewer is tired",
    "language": "French",
    "followup_message": "Nous sommes désolés que vous soyez fatigué..."
}
```

In [23]:
overall_chain.invoke({"Review": "Je suis fatigué"})

{'English_Review': 'I am tired',
 'summary': 'The reviewer expressed exhaustion and fatigue.',
 'language': 'French',
 'followup_message': "Je suis désolé d'apprendre que vous vous sentez épuisé. Prenez le temps de vous reposer et de prendre soin de vous. Peut-être une petite pause vous aidera à retrouver votre énergie et votre motivation. N'hésitez pas à parler à quelqu'un si vous en ressentez le besoin. Bon courage!"}

## Router Chain

In [24]:
physics_template = """You are a very smart physics professor.
You are great at answering questions about physics in a concise and easy to understand manner.
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}"""

math_template = """You are a very good mathematician.
You are great at answering math questions.
You are so good because you are able to break down hard problems into their component parts, 
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian.
You have an excellent knowledge of and understanding of people, 
events and contexts from a range of historical periods.
You have the ability to think, reflect, debate, discuss and evaluate the past. 
You have a respect for historical evidence and the ability to make use of it to support your explanations
and judgements.

Here is a question:
{input}"""

computerscience_template = """ You are a successful computer scientist. 
You have a passion for creativity, collaboration, 
forward-thinking, confidence, strong problem-solving capabilities, 
understanding of theories and algorithms, and excellent communication 
skills. You are great at answering coding questions.
You are so good because you know how to solve a problem by 
describing the solution in imperative steps 
that a machine can easily interpret and you know how to 
choose a solution that has a good balance between 
time complexity and space complexity. 

Here is a question:
{input}"""

In [25]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [26]:
llm = ChatOpenAI(temperature=0, model=llm_model)

In [27]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    destination_chains[name] = prompt | llm | StrOutputParser()  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [28]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = default_prompt | llm | StrOutputParser()

In [29]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a 
language model select the model prompt best suited for the input.
You will be given the names of the available prompts and a
description of what the prompt is best suited for.
You may also revise the original input if you think that revising 
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of the candidate prompts listed below.
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [30]:
import json, re
from langchain_core.prompts import PromptTemplate

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"]
)

router_chain = router_prompt | llm | StrOutputParser()

In [31]:
def parse_router_output(text):
    match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    return {"destination": "DEFAULT", "next_inputs": text}

In [32]:
def chain(input_text):
    router_output = router_chain.invoke({"input": input_text})
    parsed = parse_router_output(router_output)
    
    destination = parsed.get("destination", "DEFAULT")
    next_input = parsed.get("next_inputs", input_text)
    
    if destination in destination_chains:
        return destination_chains[destination].invoke({"input": next_input})
    return default_chain.invoke({"input": next_input})

In [33]:
chain("What is black body radiation?")

"Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This radiation is characterized by a continuous spectrum of wavelengths and intensities, with the peak intensity shifting to shorter wavelengths as the temperature of the black body increases. Black body radiation plays a key role in understanding concepts such as thermal radiation and the quantization of energy in quantum mechanics."

In [34]:
chain("what is 2 + 2")

'Thank you for the compliment! The answer to the question "what is 2 + 2" is 4.'

In [35]:
chain("Why does every cell in our body contain DNA?")

'Every cell in our body contains DNA because DNA carries the genetic information that determines the characteristics and functions of an organism. DNA contains the instructions for building and maintaining an organism, including the proteins that are essential for cell structure and function. This genetic information is passed down from parent to offspring and is essential for the growth, development, and functioning of all cells in the body. Having DNA in every cell ensures that each cell has the necessary information to carry out its specific functions and contribute to the overall functioning of the organism.'